**SENTIMENTAL ANALYSIS**

In [16]:
import pandas as pd
df=pd.read_csv('/content/IMDB Dataset.csv', encoding='latin1')

In [17]:
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [20]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0,
})

In [21]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


In [25]:
import re
negation_words=[
    "not good","not bad","not great","don't like","didn't like","never liked","wasn't good","isn't good","no good"
]

def clean_text(text):
  text=text.lower()

  text=re.sub(r"[^a-zA-Z/s']"," ",text)

  #convert negations into single tokens
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))

  return text


In [26]:
clean_text("This film is not good")

'this film is not_good'

In [27]:
df['review']=df['review'].apply(clean_text)

In [28]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)



In [29]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size=20000
max_len=250

tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV")
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)

X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.1),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [33]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [34]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
history=model.fit(
    X_train_pad,
    y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 372s 734ms/step - accuracy: 0.5515 - loss: 0.6738 - val_accuracy: 0.5867 - val_loss: 0.6244
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 383s 734ms/step - accuracy: 0.6267 - loss: 0.6123 - val_accuracy: 0.5854 - val_loss: 0.6414
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 369s 737ms/step - accuracy: 0.7753 - loss: 0.4872 - val_accuracy: 0.8249 - val_loss: 0.4510
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 387s 747ms/step - accuracy: 0.7921 - loss: 0.4412 - val_accuracy: 0.5886 - val_loss: 0.6253
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 415s 829ms/step - accuracy: 0.8339 - loss: 0.3599 - val_accuracy: 0.8720 - val_loss: 0.3197


In [36]:
loss,acc=model.evaluate(X_test_pad,y_test)
print(f"Test Accuracy: {acc:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 29s 92ms/step - accuracy: 0.8704 - loss: 0.3209
Test Accuracy: 0.8704


In [37]:
def predict_sentiment(review):
  review=clean_text(review)

  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding="post")
  prediction=model.predict(padded)[0][0]

  print("\nReview:",review)
  print("Score:",prediction)

  if prediction>=0.5:
    print("Sentiment:Positive")
  else:
    print("Sentiment:Negative")

In [42]:
predict_sentiment("film is good")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step

Review: film is good
Score: 0.7274071
Sentiment:Positive
